# FaST-LMM: full mouse dataset with LOCO

Run `single_snp` on all 1,940 individuals and 10,150 SNPs across 20 chromosomes.
Leave-one-chromosome-out (LOCO) uses the other chromosomes to build each background kernel;
rank is chosen automatically. The phenotype comes from the supplied example file.

Use a kernel with this FaST-LMM checkout and its CuPy extra installed.
PySnpTools reads BED data with CPU-only bed-reader and transfers it to CuPy.

Change `xp` between `cp` and `np`, and `ARRAY_DTYPE` in the first cell between `"float32"` and `"float64"`.
The first GPU run includes initialization and may include compilation; repeat the run cell
for a warmed-up timing. Data lookup and metadata setup happen before the timer.

`single_snp` uses the notebook's `ARRAY_DTYPE` setting for nested reads and workers, and restores the prior environment afterward. Explicit dtype arguments override that setting. Internal statistical calculations retain their required precision.


In [ ]:
import os

# Default precision for this notebook; explicit dtype arguments can override it.
os.environ["ARRAY_DTYPE"] = "float32"

import time

import cupy as cp
import numpy as np
from pysnptools.snpreader import Bed

from fastlmm.association import single_snp
from fastlmm.util import example_file

from pysnptools.util.gpu import has_gpu_support
print("Usable CUDA device:", has_gpu_support())


In [ ]:
xp = cp  # cp = GPU, np = CPU
if xp is cp and not has_gpu_support():
    raise RuntimeError("GPU timing requires a usable CUDA device")

bed = Bed(example_file("tests/datasets/mouse/alldata.*", "*.bed"), count_A1=False)
pheno = example_file("tests/datasets/mouse/pheno_10_causals.txt")
chromosome_count = len(np.unique(bed.pos[:, 0]))
print(f"{bed.iid_count:,} individuals, {bed.sid_count:,} SNPs, {chromosome_count} chromosomes")

start = time.perf_counter()
results = single_snp(
    test_snps=bed,
    pheno=pheno,
    leave_out_one_chrom=True,
    xp=xp,
)
seconds = time.perf_counter() - start

print(f"{xp.__name__} {os.environ['ARRAY_DTYPE']}: {seconds:.2f} s")
results.head(10)